In [1]:
!pip install xgboost

In [2]:
# ==========================================
# IMPORTS
# ==========================================

import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from xgboost import XGBRegressor

# ==========================================
# LOAD DATASET
# ==========================================

housing = fetch_california_housing(as_frame=True)

df = housing.frame

print("Dataset Shape:", df.shape)

# ==========================================
# FEATURES & TARGET
# ==========================================

X = df.drop("MedHouseVal", axis=1)

y = df["MedHouseVal"]

# ==========================================
# TRAIN TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==========================================
# HYPERPARAMETER TUNING FUNCTION
# ==========================================

def find_best_xgboost(X_train, y_train):

    param_grid = {

        "n_estimators": [100, 200],

        "max_depth": [3, 5, 7],

        "learning_rate": [0.05, 0.1],

        "subsample": [0.8, 1.0],

        "colsample_bytree": [0.8, 1.0]
    }

    xgb = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    grid = GridSearchCV(
        estimator=xgb,
        param_grid=param_grid,
        cv=5,
        scoring="r2",
        verbose=1,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    return (
        grid.best_estimator_,
        grid.best_params_,
        grid.best_score_
    )

# ==========================================
# FIND BEST MODEL
# ==========================================

best_model, best_params, best_cv_score = find_best_xgboost(
    X_train,
    y_train
)

print("\nBest Parameters")
print(best_params)

print("\nBest CV Score")
print(round(best_cv_score, 4))

# ==========================================
# TRAIN BEST MODEL
# ==========================================

best_model.fit(X_train, y_train)

# ==========================================
# PREDICTIONS
# ==========================================

y_train_pred = best_model.predict(X_train)

y_test_pred = best_model.predict(X_test)

# ==========================================
# EVALUATION
# ==========================================

train_r2 = r2_score(y_train, y_train_pred)

test_r2 = r2_score(y_test, y_test_pred)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_test_pred
    )
)

mae = mean_absolute_error(
    y_test,
    y_test_pred
)

print("\n" + "="*50)

print("FINAL MODEL RESULTS")

print("="*50)

print(f"Train R² : {train_r2:.4f}")

print(f"Test R²  : {test_r2:.4f}")

print(f"RMSE      : {rmse:.4f}")

print(f"MAE       : {mae:.4f}")

# ==========================================
# FEATURE IMPORTANCE
# ==========================================

importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": best_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance")

print(importance)

Dataset Shape: (20640, 9)
Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best Parameters
{'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}

Best CV Score
0.8446

FINAL MODEL RESULTS
Train R² : 0.9573
Test R²  : 0.8475
RMSE      : 0.4470
MAE       : 0.2922

Feature Importance
      Feature  Importance
0      MedInc    0.299682
7   Longitude    0.180227
6    Latitude    0.146595
2    AveRooms    0.124816
5    AveOccup    0.124117
1    HouseAge    0.054286
3   AveBedrms    0.047628
4  Population    0.022648
